# Ders 1: Python ile İlk Haritam 🗺️

Bu defter (notebook) hücrelerden oluşur:
- **Yazı hücreleri** (bunun gibi) açıklama içerir.
- **Kod hücreleri** gri kutulardır. Birine tıklayıp **Shift + Enter** basınca çalışır, sonucu altında görünür.

Hücreleri **yukarıdan aşağıya sırayla** çalıştırın. Bir şey bozulursa üst menüden *Kernel → Restart Kernel and Clear Outputs* deyip baştan başlayın — hiçbir şey kalıcı olarak bozulmaz.

Sağ üstte **"Python (CBS)"** yazmalı. Yazmıyorsa oraya tıklayıp onu seçin.

## 1. Kütüphaneleri yüklemek

Python'da her iş için hazır "alet çantaları" (kütüphaneler) vardır:
- **geopandas** → vektör veriyi (nokta, çizgi, alan) okur ve işler. Excel tablosu gibi ama her satırın bir *geometrisi* var.
- **matplotlib** → grafik/harita çizer.
- **folium** → internette gezinilebilir (yakınlaştırılabilir) harita yapar.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
print("Hazırız! geopandas sürümü:", gpd.__version__)

## 2. Vektör veriyi okumak

`veri/turkiye_iller.geojson` dosyası 81 ilin sınırlarını içeriyor (kaynak: geoBoundaries / OpenStreetMap).

**GeoJSON** nedir? Coğrafi veriyi düz metin olarak saklayan yaygın bir dosya biçimidir. Diğer yaygın biçimler: **Shapefile (.shp)**, **GeoPackage (.gpkg)**.

In [ ]:
iller = gpd.read_file("veri/turkiye_iller.geojson")
iller.head()   # ilk 5 satırı göster

👆 Tabloya bakın: her satır bir il. En sağdaki **geometry** sütunu ilin sınır şeklini tutar (POLYGON / MULTIPOLYGON = alan).

Bu tabloya CBS'de **öznitelik tablosu** (attribute table) denir. QGIS'te bir katmana sağ tıklayıp *Open Attribute Table* dediğinizde aynısını görürsünüz.

In [ ]:
print("Kaç il var?", len(iller))
print("Geometri türleri:", iller.geom_type.unique())

## 3. Koordinat Referans Sistemi (KRS / CRS) — en önemli konu!

Her coğrafi verinin bir KRS'si vardır: sayıların *ne anlama geldiğini* söyler.

In [ ]:
iller.crs

**EPSG:4326 (WGS 84)** = enlem/boylam, birim **derece**. Google Earth ve GPS bunu kullanır.

⚠️ Derece ile alan/mesafe hesaplanmaz (1 derece boylam ekvatorda ~111 km, kutupta 0 km!). Önce **metre** kullanan bir KRS'ye **dönüştürmemiz (reproject)** gerekir.

Türkiye geneli için **EPSG:5254 (TUREF / TM33)** gibi yerel sistemler veya tek bir **UTM** dilimi kullanılabilir. Bütün ülke için alan hesaplarken eşit-alan projeksiyonu daha doğrudur; burada **EPSG:6933** (dünya eşit alan) kullanalım.

In [ ]:
iller_m = iller.to_crs(epsg=6933)          # metreye çevir
iller_m["alan_km2"] = iller_m.area / 1_000_000   # m² → km²
iller_m[["shapeName", "alan_km2"]].sort_values("alan_km2", ascending=False).head(10)

🎯 Konya en büyük il çıkmalı (~41.000 km²). Resmi rakamlarla küçük farklar normaldir: veri **basitleştirilmiş** (simplified) sınırlar kullanıyor.

**Alıştırma:** Aşağıdaki hücreyi değiştirerek en *küçük* 5 ili bulun. İpucu: `ascending=True`

In [ ]:
# Buraya yazın:


## 4. İlk haritam

`.plot()` tek satırda haritayı çizer. `column=` ile illeri bir değere göre renklendirebiliriz — buna **tematik harita** (koroplet) denir.

In [ ]:
ax = iller_m.plot(column="alan_km2", cmap="YlGn", legend=True,
                  edgecolor="gray", linewidth=0.3, figsize=(12, 6),
                  legend_kwds={"label": "Alan (km²)", "shrink": 0.6})
ax.set_title("Türkiye illeri — yüzölçümü")
ax.set_axis_off()
plt.show()

**Alıştırma:** `cmap="YlGn"` yerine `"viridis"`, `"OrRd"` veya `"Blues"` deneyin. Renk paleti seçimi haritanın okunabilirliğini çok etkiler.

## 5. Tablodan nokta katmanı yapmak (CSV → nokta)

Çoğu zaman elinizde enlem/boylam sütunları olan bir Excel/CSV olur. Bunu haritaya dökmek CBS'nin en sık işlerinden biridir.

In [ ]:
tablo = pd.read_csv("veri/sehirler.csv")
tablo

In [ ]:
sehirler = gpd.GeoDataFrame(
    tablo,
    geometry=gpd.points_from_xy(tablo["boylam"], tablo["enlem"]),  # DİKKAT: önce X=boylam, sonra Y=enlem!
    crs="EPSG:4326",
)
sehirler.head()

⚠️ **Klasik hata:** `points_from_xy(x, y)` → x = **boylam**, y = **enlem**. Ters yazarsanız noktalar Hint Okyanusu'na düşer. Deneyin ve görün! 😄

In [ ]:
ax = iller.plot(color="whitesmoke", edgecolor="gray", linewidth=0.3, figsize=(12, 6))
sehirler.plot(ax=ax, color="crimson", markersize=40)
for _, r in sehirler.iterrows():
    ax.annotate(r["ad"], (r.geometry.x, r.geometry.y), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_axis_off(); plt.show()

Burada iki katmanı **üst üste** koyduk. İkisinin de KRS'si EPSG:4326 olduğu için çakıştılar. KRS'ler farklı olsaydı önce `.to_crs()` ile eşitlememiz gerekirdi.

## 6. Mekânsal birleştirme (spatial join) — "hangi nokta hangi ilin içinde?"

Bu, CBS'yi sıradan tablolardan ayıran şeydir: tablolar ortak bir *sütunla* değil, **konumla** birleşir.

In [ ]:
sonuc = gpd.sjoin(sehirler, iller[["shapeName", "geometry"]], predicate="within")
sonuc[["ad", "tur", "shapeName"]].rename(columns={"shapeName": "bulundugu_il"})

🎯 "Kapadokya (Göreme)" → Nevşehir, "Ağrı Dağı" → Ağrı (veya Iğdır sınırına yakın) çıkmalı. Python bunu sadece koordinatlara bakarak buldu!

## 7. Etkileşimli web haritası

**folium** ile yakınlaştırılabilir, tıklanabilir bir harita yapalım. Altlık harita OpenStreetMap'ten gelir (internet gerekir).

In [ ]:
m = folium.Map(location=[39, 35], zoom_start=6, tiles="OpenStreetMap")
folium.GeoJson(iller, name="İller",
               style_function=lambda f: {"fillOpacity": 0.05, "color": "#555", "weight": 1},
               tooltip=folium.GeoJsonTooltip(fields=["shapeName"], aliases=["İl:"])).add_to(m)
for _, r in sehirler.iterrows():
    folium.Marker([r["enlem"], r["boylam"]], popup=f"{r['ad']} ({r['tur']})").add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
m.save("benim_haritam.html")
print("Kaydedildi: CBS-Ogrenme/benim_haritam.html — çift tıklayıp tarayıcıda açabilirsiniz.")

## 8. Sonuçları kaydetmek (QGIS'te açmak için)

**GeoPackage (.gpkg)** modern, tek dosyalık, önerilen biçimdir. Birden çok katmanı tek dosyada tutabilir.

In [ ]:
iller_m.to_file("veri/calisma.gpkg", layer="iller_alan", driver="GPKG")
sehirler.to_file("veri/calisma.gpkg", layer="sehirler", driver="GPKG")
print("Kaydedildi: veri/calisma.gpkg → QGIS'e sürükleyip bırakın!")

## 🎓 Ders 1 bitti!

Öğrendikleriniz: vektör veri, öznitelik tablosu, **KRS ve dönüşüm**, alan hesabı, tematik harita, CSV'den nokta, mekânsal birleştirme, web haritası, GeoPackage.

**Sıradaki adım:** `REHBER.md` dosyasındaki QGIS alıştırmalarını yapın — aynı işleri fareyle yapmayı öğreneceksiniz. Takıldığınız her yerde Claude'a sorun.